# Entry-Level Data Engineering Python Interview Practice

**20 questions · 4 datasets · Core Python only**

This notebook focuses on practical Python used in junior data-engineering interviews: filtering, aggregation, parsing, validation, deduplication, timestamps, incremental processing, and mini-ETL design.

### Recommended workflow

1. Work on one or two questions per session.
2. Do not use Pandas unless you intentionally want a second solution.
3. Run the dataset cell at the start of each batch.
4. Explain your time/space complexity and edge cases aloud.
5. Replace `raise NotImplementedError` with your solution.

### GitHub Codespaces

Open this repository in a Codespace, select this notebook, choose a Python kernel, then use **Run Cell**. No third-party packages are required.

> The notebook intentionally contains prompts and starter code, not solutions. Dataset inputs are copied only when the problem requires non-mutation; your functions should still honor each stated constraint.

## Progress tracker

- [ ] Q1  - [ ] Q2  - [ ] Q3  - [ ] Q4  - [ ] Q5
- [ ] Q6  - [ ] Q7  - [ ] Q8  - [ ] Q9  - [ ] Q10
- [ ] Q11 - [ ] Q12 - [ ] Q13 - [ ] Q14 - [ ] Q15
- [ ] Q16 - [ ] Q17 - [ ] Q18 - [ ] Q19 - [ ] Q20

In [ ]:
# Standard-library imports you may find useful
from collections import Counter, defaultdict
from datetime import datetime
from pprint import pprint
import re

# Batch 1 — E-commerce Orders

Skills: filtering, arithmetic, dictionary aggregation, nested summaries, and selecting a maximum.

In [ ]:
orders = [
    {"order_id": 1001, "customer_id": "C001", "product": "Laptop",   "category": "Electronics", "quantity": 1, "unit_price": 1200.00, "status": "completed"},
    {"order_id": 1002, "customer_id": "C002", "product": "Mouse",    "category": "Electronics", "quantity": 2, "unit_price": 25.00,   "status": "completed"},
    {"order_id": 1003, "customer_id": "C001", "product": "Keyboard", "category": "Electronics", "quantity": 1, "unit_price": 75.00,   "status": "cancelled"},
    {"order_id": 1004, "customer_id": "C003", "product": "Desk",     "category": "Furniture",   "quantity": 1, "unit_price": 300.00,  "status": "completed"},
    {"order_id": 1005, "customer_id": "C002", "product": "Chair",    "category": "Furniture",   "quantity": 4, "unit_price": 150.00,  "status": "completed"},
    {"order_id": 1006, "customer_id": "C004", "product": "Monitor",  "category": "Electronics", "quantity": 2, "unit_price": 400.00,  "status": "completed"},
    {"order_id": 1007, "customer_id": "C003", "product": "Mouse",    "category": "Electronics", "quantity": 1, "unit_price": 25.00,   "status": "cancelled"},
    {"order_id": 1008, "customer_id": "C001", "product": "Monitor",  "category": "Electronics", "quantity": 1, "unit_price": 400.00,  "status": "completed"},
]

len(orders)

### Q1 — Filter records ★

Return only records where `status == "completed"`. **Do not modify the original list.**

In [ ]:
def get_completed_orders(orders):
    # TODO: implement your solution
    return [order for order in orders if order['status'] == 'completed']

print(get_completed_orders(orders))

### Q2 — Calculate revenue ★

Calculate total revenue from completed orders only, where `revenue = quantity × unit_price`. Return a `float`.

In [ ]:
def calculate_total_revenue(orders):
    # TODO: implement your solution
    return sum([(order['quantity'] * order['unit_price']) for order in orders if order['status'] == 'completed'])
    raise NotImplementedError

calculate_total_revenue(orders)

### Q3 — Aggregate by customer ★★

Return `{customer_id: completed_revenue}`. Do not hard-code customer IDs.

In [ ]:
def revenue_by_customer(orders):
    result = {}

    for order in orders:
        if order["status"] != "completed":
            continue

        customer_id = order["customer_id"]
        revenue = order["quantity"] * order["unit_price"]

        if customer_id not in result:
            result[customer_id] = 0

        result[customer_id] += revenue

    return result


revenue_by_customer(orders)

### Q4 — Aggregate by category ★★

Return a nested dictionary for each category with `orders`, `units_sold`, and `revenue`. Cancelled orders contribute nothing. Be ready to explain the difference between order count and units sold.

In [ ]:
def category_summary(orders):
    # TODO: implement your solution
    summary = {}

    for order in orders:
        if order['status'] != 'completed':
            continue

        category = order['category']
        
        if category not in summary:
            summary[category] = {
                "orders":0,
                "units_sold":0,
                "revenue":0
            }

        summary[category]['orders'] += 1
        summary[category]['units_sold'] += order['quantity']
        summary[category]['revenue'] += order['quantity'] * order['unit_price']

    return summary

    raise NotImplementedError

print(category_summary(orders))

### Q5 — Top customer ★★

Return `(customer_id, revenue)` for the customer with the highest completed revenue. Do not assume the most orders means the most revenue. Decide what an empty input should return.

In [ ]:
def top_customer(orders):
    # TODO: implement your solution

    orders = [order for order in orders if order['status'] == 'completed']

    top_customer = ['none',0]
    result = {}
    for order in orders:
        customer_id = order['customer_id']

        if customer_id not in result:
            result[customer_id] = 0

        revenue = order['unit_price'] * order['quantity']

        result[customer_id] += revenue
        
        if result[customer_id] > top_customer[1]:
            top_customer[0] = customer_id
            top_customer[1] = result[customer_id]

    return (top_customer[0],top_customer[1])
    
print(top_customer(orders))



# Batch 2 — Application Logs

Skills: raw-to-structured parsing, counting, threshold filtering, regular expressions, and timestamp arithmetic.

Each raw line follows `timestamp|level|service|message`.

In [ ]:
logs = [
    "2026-09-01 10:00:01|INFO|user_service|User 101 logged in",
    "2026-09-01 10:00:03|ERROR|payment_service|Payment failed for user 205",
    "2026-09-01 10:00:05|WARNING|inventory_service|Low inventory for SKU A12",
    "2026-09-01 10:01:10|ERROR|payment_service|Payment failed for user 309",
    "2026-09-01 10:02:15|INFO|payment_service|Payment completed for user 205",
    "2026-09-01 10:03:20|ERROR|user_service|Database connection failed",
    "2026-09-01 10:04:00|INFO|inventory_service|Inventory updated for SKU B55",
    "2026-09-01 10:05:12|ERROR|payment_service|Payment failed for user 205",
    "2026-09-01 10:06:30|WARNING|user_service|Slow response detected",
    "2026-09-01 10:07:45|INFO|user_service|User 309 logged out",
]

len(logs)

### Q6 — Parse raw records ★

Convert each raw string into a dictionary with `timestamp`, `level`, `service`, and `message`. Split only as much as needed so a message containing `|` can still be retained.

In [ ]:
def parse_logs(logs):
    # TODO: implement your solution
    records = []

    for log in logs:
        timestamp = log[:19]
        info = log[20:]
        split_info = info.split('|')

        record = {
            'timestamp': timestamp,
            'level': split_info[0],
            'service': split_info[1],
            'message': split_info[2]
        }

        records.append(record)
    
    return records

print(parse_logs(logs))

### Q7 — Count errors ★★

Using parsed records, return the number of `ERROR` events per service. Do not include services with zero errors.

In [ ]:
def error_count_by_service(records):
    # TODO: implement your solution
    error_tracker = {}

    for record in records:
        service = record['service']

        if record['level'] == 'ERROR':
            error_tracker[service] = (
                error_tracker.get(service,0) + 1
            )
    
    return error_tracker

print(error_count_by_service(parse_logs(logs)))

### Q8 — Find error-prone services ★★

Return services whose error count is greater than or equal to `threshold`. Choose and document a deterministic return order.

In [ ]:
def services_above_error_threshold(records, threshold):
    # TODO: implement your solution
    keys = records.keys()
    
    return [key for key in keys if records[key] >= threshold]


print(services_above_error_threshold(error_count_by_service(parse_logs(logs)),1))



### Q9 — Extract information from text ★★

Find every user ID associated with a failed payment. Return unique IDs as integers, sorted ascending. Do not assume IDs have a fixed number of digits.

In [ ]:
def failed_payment_user_ids(records):
    # TODO: implement your solution
    user_ids = []
    for record in records:
        message = record['message']

        if 'Payment failed' not in message:
            continue

        match = re.search(r'user (\d+)',message)

        if not match:
            continue

        user_id = int(match.group(1))

        if user_id not in user_ids:
            user_ids.append(user_id)
    
    return user_ids

print(failed_payment_user_ids(parse_logs(logs)))

### Q10 — Time between errors ★★★

Find the difference in seconds between consecutive `ERROR` events. Return dictionaries with `previous_error`, `current_error`, and `seconds_between`. Parse timestamps with `datetime`; do not rely on lexical subtraction.

In [ ]:
def seconds_between_errors(records):
    # TODO: implement your solution
    result = []
    error_records = [
        record
        for record in records
        if record['level'] == 'ERROR'
    ]

    error_records = sorted(
        error_records,
        key = lambda record: datetime.strptime(
            record['timestamp'],
            '%Y-%m-%d %H:%M:%S'
        )
    )
    
    previous_error, current_error = None, None

    for record in error_records:

        previous_error = current_error    
        current_error = datetime.strptime(record['timestamp'], '%Y-%m-%d %H:%M:%S')

        if previous_error:
            difference = current_error - previous_error
            seconds = difference.total_seconds()

            error_event = {
                'previous_error': previous_error.strftime('%Y-%m-%d %H:%M:%S'),
                'current_error': current_error.strftime('%Y-%m-%d %H:%M:%S'),
                'seconds_between': int(seconds)
            }

            result.append(error_event)

    return result

records = parse_logs(logs)

seconds_between_errors(records)

# Batch 3 — Dirty Customer Data

Skills: normalization, multi-format date validation, quarantine patterns, deterministic deduplication, and data-quality measurement.

In [ ]:
customers = [
    {"customer_id": "C001", "name": "  Alice Kim ", "email": "ALICE@EXAMPLE.COM", "country": "kr",           "signup_date": "2026-01-15"},
    {"customer_id": "C002", "name": "bob lee",       "email": "bob@example.com",   "country": "South Korea",  "signup_date": "2026/02/20"},
    {"customer_id": "C003", "name": "Charlie Park",  "email": None,                "country": "KR",           "signup_date": "2026-02-30"},
    {"customer_id": "C001", "name": "Alice Kim",     "email": "alice@example.com", "country": "Korea",        "signup_date": "2026-01-15"},
    {"customer_id": "C004", "name": "DANA CHOI",     "email": "dana@example",      "country": "USA",          "signup_date": "2026-03-10"},
    {"customer_id": "C005", "name": " Evan Jung ",   "email": "evan@example.com",  "country": "us",           "signup_date": "03-15-2026"},
    {"customer_id": "C006", "name": "",              "email": "frank@example.com", "country": "United States", "signup_date": "2026-04-01"},
]

len(customers)

### Q11 — Normalize strings ★★

Return a new customer dictionary. Trim and title-case `name`; lowercase `email`; map `kr`, `KR`, `Korea`, and `South Korea` to `KR`; map `USA`, `us`, and `United States` to `US`. Do not mutate the input.

In [ ]:
def normalize_customer(customer):
    # TODO: implement your solution
    normalized = customer.copy()
    normalized['name'] = normalized['name'].strip().title()
    normalized['email'] = normalized['email'].lower() if normalized['email'] else None

    country_map = {
        'kr':'South Korea',
        'KR':'South Korea',
        'Korea':'South Korea',
        'USA':'US',
        'us':'US',
        'United States':'US'
    }

    normalized['country'] = country_map.get(normalized['country'],normalized['country'])

    return normalized

normalize_customer(customers[4])

### Q12 — Validate records ★★

Return `True` only when the record has a non-empty customer ID and name, a non-null email containing both `@` and `.`, and a real signup date. Accept `YYYY-MM-DD`, `YYYY/MM/DD`, or `MM-DD-YYYY`; let `datetime` reject impossible dates.

In [ ]:
def validate_customer(customer):
    # TODO: implement your solution

    if customer.get('customer_id','') == '' or customer.get('name','') == '':
        return False

    email = customer.get('email') or ''

    if not ('.' in email and '@' in email):
        return False

    accepted_date_formats = [
        '%Y-%m-%d',
        '%Y/%m/%d',
        '%m-%d-%Y'
    ]

    valid_date = False
    for format in accepted_date_formats:
        try:
            signup_date = customer.get('signup_date','') or ''
            datetime.strptime(signup_date,format)
            valid_date = True
            break
        except ValueError:
            continue

    return valid_date


### Q13 — Separate good and bad records ★★

Return `(valid_records, rejected_records)`. Preserve input order and retain bad records instead of silently discarding them.

In [ ]:
def split_records(customers):
    # TODO: implement your solution
    valid_records = []
    rejected_records = []

    for customer in customers:
        if validate_customer(customer):
            valid_records.append(customer)
        else:
            rejected_records.append(customer)

    return (valid_records, rejected_records)

split_records(customers)

### Q14 — Deduplicate customers ★★★

Keep one record per `customer_id`. Prefer a valid record over an invalid one; if both are equally valid, keep the record appearing later. Preserve a deterministic output order and explain your choice.

In [ ]:
def deduplicate_customers(customers):
    # TODO: implement your solution
    best_by_customer_id = {}
    for customer in customers:
        customer_id = customer['customer_id']
        existing_record = best_by_customer_id.get(customer_id)
        current_is_valid = validate_customer(customer)

        if existing_record is None:
            best_by_customer_id[customer_id] = {
                'customer': customer,
                'valid': validate_customer(customer)
            }
            continue

        existing_is_valid = existing_record['valid']

        if not existing_is_valid or current_is_valid:
            best_by_customer_id[customer_id] = {
                "customer": customer,
                "valid": current_is_valid,
            }

    return [
        record['customer']
        for record in best_by_customer_id.values()
    ]

deduplicate_customers(customers)


### Q15 — Build a quality report ★★★

Return counts for `total_records`, `valid_records`, `invalid_records`, `duplicate_customer_ids`, `missing_names`, `missing_emails`, `invalid_dates`, and `invalid_emails`. A record may contribute to multiple problems. Define whether duplicate IDs means repeated rows or distinct IDs that repeat.

In [ ]:
def is_valid_signup_date(signup_date):
    accepted_date_formats = [
        '%Y-%m-%d',
        '%Y/%m/%d',
        '%m-%d-%Y'
    ]

    for format in accepted_date_formats:
        try:
            datetime.strptime(signup_date, format)
            return True
        except ValueError:
            continue

    return False

        
def data_quality_report(customers):
    # TODO: implement your solution
    total_records = len(customers)

    valid_records, invalid_records = split_records(customers)

    duplicate_customer_ids = []
    duplicate_count = 0
    missing_names = 0
    missing_emails = 0
    invalid_emails = 0
    invalid_dates = 0

    for customer in customers:
        email = customer.get('email') or ''
        name = customer.get('name') or ''

        if email == '':
            missing_emails += 1

        if not ('.' in email and '@' in email):
            invalid_emails += 1

        if name.strip() == '':
            missing_names += 1

        if customer['customer_id'] in duplicate_customer_ids:
            duplicate_count += 1
        else:
            duplicate_customer_ids.append(customer['customer_id'])

        if not is_valid_signup_date(customer.get('signup_date') or ''):
            invalid_dates += 1

    return {
        'total_records':total_records,
        'valid_records':len(valid_records),
        'invalid_records':len(invalid_records),
        'duplicate_customer_ids':duplicate_count,
        'missing_names':missing_names,
        'missing_emails':missing_emails,
        'invalid_dates':invalid_dates,
        'invalid_emails':invalid_emails
    }

data_quality_report(customers)


# Batch 4 — Event Pipeline

Skills: idempotency, event aggregation, ordered funnel logic, watermarks, late-arriving data, and end-to-end batch processing.

In [ ]:
events = [
    {"event_id": "E001", "user_id": 101, "event_type": "view",     "product_id": "P10", "timestamp": "2026-09-01T10:00:00"},
    {"event_id": "E002", "user_id": 101, "event_type": "cart",     "product_id": "P10", "timestamp": "2026-09-01T10:02:00"},
    {"event_id": "E003", "user_id": 205, "event_type": "view",     "product_id": "P20", "timestamp": "2026-09-01T10:03:00"},
    {"event_id": "E004", "user_id": 101, "event_type": "purchase", "product_id": "P10", "timestamp": "2026-09-01T10:05:00"},
    {"event_id": "E005", "user_id": 205, "event_type": "cart",     "product_id": "P20", "timestamp": "2026-09-01T10:06:00"},
    {"event_id": "E003", "user_id": 205, "event_type": "view",     "product_id": "P20", "timestamp": "2026-09-01T10:03:00"},
    {"event_id": "E006", "user_id": 309, "event_type": "view",     "product_id": "P30", "timestamp": "2026-09-01T10:07:00"},
    {"event_id": "E007", "user_id": 205, "event_type": "purchase", "product_id": "P20", "timestamp": "2026-09-01T10:10:00"},
    {"event_id": "E008", "user_id": 101, "event_type": "view",     "product_id": "P20", "timestamp": "2026-09-01T10:12:00"},
    {"event_id": "E009", "user_id": 309, "event_type": "cart",     "product_id": "P30", "timestamp": "2026-09-01T10:15:00"},
    {"event_id": "E010", "user_id": 101, "event_type": "purchase", "product_id": "P20", "timestamp": "2026-09-01T10:20:00"},
]

len(events)
events

### Q16 — Deduplicate events ★★

Remove duplicate records by `event_id`, preserving the order in which each unique ID first appeared. Do not modify the original list.

In [8]:
def deduplicate_events(events):
    # TODO: implement your solution
    events_copy = []
    event_ids = set()
    for event in events:
        event_id = event.get('event_id') or ''

        if event_id in event_ids:
            continue

        events_copy.append(event)
        event_ids.add(event_id)

    return events_copy

deduplicate_events(events)

[{'event_id': 'E001',
  'user_id': 101,
  'event_type': 'view',
  'product_id': 'P10',
  'timestamp': '2026-09-01T10:00:00'},
 {'event_id': 'E002',
  'user_id': 101,
  'event_type': 'cart',
  'product_id': 'P10',
  'timestamp': '2026-09-01T10:02:00'},
 {'event_id': 'E003',
  'user_id': 205,
  'event_type': 'view',
  'product_id': 'P20',
  'timestamp': '2026-09-01T10:03:00'},
 {'event_id': 'E004',
  'user_id': 101,
  'event_type': 'purchase',
  'product_id': 'P10',
  'timestamp': '2026-09-01T10:05:00'},
 {'event_id': 'E005',
  'user_id': 205,
  'event_type': 'cart',
  'product_id': 'P20',
  'timestamp': '2026-09-01T10:06:00'},
 {'event_id': 'E006',
  'user_id': 309,
  'event_type': 'view',
  'product_id': 'P30',
  'timestamp': '2026-09-01T10:07:00'},
 {'event_id': 'E007',
  'user_id': 205,
  'event_type': 'purchase',
  'product_id': 'P20',
  'timestamp': '2026-09-01T10:10:00'},
 {'event_id': 'E008',
  'user_id': 101,
  'event_type': 'view',
  'product_id': 'P20',
  'timestamp': '2026-09

### Q17 — User event summary ★★

Return `{user_id: {event_type: count}}`. Duplicate event IDs must not be counted twice, and event types must not be hard-coded.

In [10]:
def user_event_summary(events):
    # TODO: implement your solution
    
    event_summary = {}

    for event in events:
        user_id = event.get('user_id') 
        event_type = event.get('event_type')

        user_info = event_summary.get(user_id)
    
        if user_info is None:
            event_summary[user_id] = {
                event_type : 1
            }

            continue

        user_info_event_type = user_info.get(event_type)

        if user_info_event_type is None:
            event_summary[user_id][event_type] = 1

        else:
            event_summary[user_id][event_type] += 1

    return event_summary

events_dedup = deduplicate_events(events)
user_event_summary(events_dedup)


{101: {'view': 2, 'cart': 1, 'purchase': 2},
 205: {'view': 1, 'cart': 1, 'purchase': 1},
 309: {'view': 1, 'cart': 1}}

### Q18 — Funnel analysis ★★★

For each user, return whether they completed `view → cart → purchase` in chronological order. Merely having all three types is insufficient. Consider whether events for different products should form one funnel, and state the assumption your implementation uses.

In [ ]:
def completed_funnel_by_user(events):
    # TODO: implement your solution
    
    event_order = {}
    user_status = {}
    event_type_to_num = {
        'view':1,
        'cart':2,
        'purchase':3
    }
    for event in events:
        user_id = event.get('user_id')
        event_type = event_type_to_num[
            event.get('event_type')
            ]
        
        existing_user_info = event_order.get(user_id)

        if existing_user_info is None:
            event_order[user_id] = [0]

            user_status[user_id] = {
                'completed' : False
            }

            if event_type == 1:
                event_order[user_id].append(event_type)

            continue

        last_event = max(event_order[user_id])

        if event_type - 1 == last_event:
            event_order[user_id].append(event_type)

        else:
            if event_type == 1:
                event_order[user_id] = [0,1]
            else: 
                event_order[user_id] = [0]

            continue

        
        if event_type == 3:
            user_status[user_id]['completed'] = True

    return user_status
        
deduped_events = deduplicate_events(events)
completed_funnel_by_user(deduped_events)



{101: {'completed': True}, 205: {'completed': True}, 309: {'completed': False}}

In [ ]:
def completed_by_user(events):
    required = [0,1,2,3]
    user_status = {}
    event_order = {}
    event_type_to_num = {
        'view':1,
        'cart':2,
        'purchase':3
    }

    for event in events:
        user_id = event.get('user_id')
        event_type = event_type_to_num[
            event.get('event_type')
        ]

        existing_user = user_status.get(user_id)
        existing_event_order = event_order.get(user_id)

        if existing_user is None:
            user_status[user_id] = False

            event_order[user_id] = set()
            event_order[user_id].add(event_type)

            continue

        existing_event_order.add(event_type)

        if all(x in existing_event_order for x in required):
            user_status[user_id] = True

            continue

    return user_status, event_order

deduped_events = deduplicate_events(events)
completed_by_user(deduped_events)

({101: False, 205: False, 309: False},
 {101: {1, 2, 3}, 205: {1, 2, 3}, 309: {1, 2}})

### Q19 — Incremental processing ★★★

Return unique events occurring **strictly after** `last_processed_timestamp`, sorted chronologically. Then answer the discussion prompt in the next cell.

In [ ]:
def get_incremental_events(events, last_processed_timestamp):
    # TODO: implement your solution
    raise NotImplementedError


In [ ]:
last_processed_timestamp = "2026-09-01T10:06:00"

# After implementing Q19:
# pprint(get_incremental_events(events, last_processed_timestamp))

#### Q19 discussion prompt

What can go wrong if the source sends a late event whose timestamp is earlier than the watermark? Describe at least one mitigation, such as an overlap/lookback window, source offset, ingestion timestamp, or reconciliation job.

**Your answer:**

> 

### Q20 — Build a mini ETL pipeline ★★★★

Implement `process_event_batch(events, processed_event_ids)`. It must: (1) require `event_id`, `user_id`, `event_type`, `product_id`, and `timestamp`; (2) reject invalid timestamps; (3) deduplicate within the batch; (4) exclude previously processed IDs; (5) sort remaining events chronologically; (6) build a user-level event summary; and (7) return cleaned records, summary, and rejected records. Do not mutate either input. Document how you classify duplicate/already-processed records.

In [ ]:
def process_event_batch(events, processed_event_ids):
    # TODO: implement your solution
    raise NotImplementedError


In [ ]:
processed_event_ids = {"E001", "E002", "E003"}

# After implementing Q20:
# result = process_event_batch(events, processed_event_ids)
# pprint(result)

# Interview debrief template

Complete this after each problem:

- **Approach:** What data structure or algorithm did I choose?
- **Correctness:** Why does it satisfy every requirement?
- **Complexity:** What are the time and space costs?
- **Edge cases:** Empty input, missing keys, malformed values, ties, duplicates?
- **Production change:** What would I add for larger or unreliable real-world data?

When you finish a question, share the function with your interviewer/reviewer rather than only the output.